# GuV-Analyse: Gewinn- und Verlustrechnung

## Gesamtkostenverfahren (GKV) & Umsatzkostenverfahren (UKV)

Dieses Notebook liest eine Excel-Datei mit zwei Tabellenblättern (GKV und UKV) ein und führt eine umfassende Analyse durch:

- **Datenimport** aus Excel (`.xlsx`) mit Spalten: `Konto | Kontenbezeichnung | Ist | Plan | Forecast`
- **Kennzahlenberechnung**: DB1, DB2, EBITDA, EBIT, EBT, Jahresüberschuss
- **Abweichungsanalyse**: Plan-Ist und Forecast-Plan (absolut & prozentual)
- **Quoten & Rentabilität**: Umsatzrentabilität, Materialquote, Personalquote
- **Visualisierungen**: Wasserfall, Abweichungsdiagramme, Kennzahlenübersicht

---

## 1. Setup & Bibliotheken

In [ ]:
# Abhängigkeiten installieren (nur beim ersten Mal nötig)
# %pip install pandas openpyxl matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Plotting-Konfiguration
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'sans-serif',
})

# Farben für konsistente Darstellung
FARBEN = {
    'ist': '#2196F3',
    'plan': '#FF9800',
    'forecast': '#4CAF50',
    'positiv': '#4CAF50',
    'negativ': '#F44336',
    'neutral': '#9E9E9E',
    'highlight': '#7B1FA2',
}

print('Bibliotheken erfolgreich geladen.')

## 2. Excel-Datei einlesen

Passe den **Dateipfad** und die **Blattnamen** an deine Excel-Datei an.

Erwartete Spalten je Blatt: `Konto | Kontenbezeichnung | Ist | Plan | Forecast`

In [ ]:
# ============================================================
# KONFIGURATION - Hier anpassen!
# ============================================================
EXCEL_PFAD = 'GuV_Daten.xlsx'           # Pfad zur Excel-Datei
BLATT_GKV = 'GKV'                       # Name des GKV-Tabellenblatts
BLATT_UKV = 'UKV'                       # Name des UKV-Tabellenblatts

# Spaltennamen in der Excel-Datei (anpassen falls abweichend)
COL_KONTO = 'Konto'
COL_BEZEICHNUNG = 'Kontenbezeichnung'
COL_IST = 'Ist'
COL_PLAN = 'Plan'
COL_FORECAST = 'Forecast'
# ============================================================

In [ ]:
def lade_guv_blatt(pfad: str, blattname: str) -> pd.DataFrame:
    """Liest ein GuV-Tabellenblatt aus einer Excel-Datei."""
    df = pd.read_excel(pfad, sheet_name=blattname)
    # Spalten bereinigen
    df.columns = df.columns.str.strip()
    # Numerische Spalten sicherstellen
    for col in [COL_IST, COL_PLAN, COL_FORECAST]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df


pfad = Path(EXCEL_PFAD)
if pfad.exists():
    df_gkv = lade_guv_blatt(EXCEL_PFAD, BLATT_GKV)
    df_ukv = lade_guv_blatt(EXCEL_PFAD, BLATT_UKV)
    print(f'GKV: {len(df_gkv)} Zeilen geladen')
    print(f'UKV: {len(df_ukv)} Zeilen geladen')
else:
    print(f'Datei "{EXCEL_PFAD}" nicht gefunden.')
    print('Verwende Beispieldaten zur Demonstration.\n')
    # Demo-Daten werden in der nächsten Zelle erzeugt
    df_gkv = None
    df_ukv = None

## 3. Beispieldaten (falls keine Excel-Datei vorhanden)

Diese Zelle erzeugt realistische Demo-Daten, damit das Notebook auch ohne Excel-Datei lauffähig ist.

In [ ]:
def erzeuge_demo_gkv() -> pd.DataFrame:
    """Erzeugt realistische GKV-Demodaten."""
    daten = [
        # Umsatzerlöse
        ('4000', 'Umsatzerlöse Inland', 8500000, 8000000, 8300000),
        ('4100', 'Umsatzerlöse Export', 3200000, 3500000, 3400000),
        ('4200', 'Erlösschmälerungen', -350000, -300000, -320000),
        ('ZS', 'Umsatzerlöse (netto)', 11350000, 11200000, 11380000),
        # Bestandsveränderungen / aktivierte Eigenleistungen
        ('4800', 'Bestandsveränderungen', 150000, 100000, 120000),
        ('4900', 'Aktivierte Eigenleistungen', 200000, 180000, 190000),
        ('ZS', 'Gesamtleistung', 11700000, 11480000, 11690000),
        # Materialaufwand
        ('5000', 'Aufwand Roh-/Hilfs-/Betriebsstoffe', -3800000, -3600000, -3750000),
        ('5100', 'Aufwand bezogene Leistungen', -950000, -900000, -920000),
        ('ZS', 'Materialaufwand', -4750000, -4500000, -4670000),
        ('ZS', 'Rohertrag / DB1', 6950000, 6980000, 7020000),
        # Personalaufwand
        ('6000', 'Löhne und Gehälter', -2800000, -2750000, -2780000),
        ('6100', 'Soziale Abgaben', -560000, -550000, -556000),
        ('6200', 'Altersversorgung', -140000, -130000, -135000),
        ('ZS', 'Personalaufwand', -3500000, -3430000, -3471000),
        ('ZS', 'DB2', 3450000, 3550000, 3549000),
        # Sonstige betriebliche Aufwendungen
        ('6300', 'Miete und Raumkosten', -480000, -470000, -475000),
        ('6400', 'Versicherungen', -120000, -110000, -115000),
        ('6500', 'Reisekosten', -95000, -80000, -85000),
        ('6600', 'Werbekosten', -210000, -200000, -205000),
        ('6700', 'IT-Kosten', -180000, -160000, -170000),
        ('6800', 'Sonstige betriebl. Aufwendungen', -265000, -250000, -255000),
        ('ZS', 'Sonstige betriebliche Aufwendungen', -1350000, -1270000, -1305000),
        # Sonstige betriebliche Erträge
        ('4500', 'Sonstige betriebliche Erträge', 180000, 150000, 165000),
        ('ZS', 'EBITDA', 2280000, 2430000, 2409000),
        # Abschreibungen
        ('6900', 'Abschreibungen Sachanlagen', -420000, -400000, -410000),
        ('6950', 'Abschreibungen immaterielle VG', -80000, -70000, -75000),
        ('ZS', 'Abschreibungen', -500000, -470000, -485000),
        ('ZS', 'EBIT', 1780000, 1960000, 1924000),
        # Finanzergebnis
        ('7000', 'Zinserträge', 25000, 20000, 22000),
        ('7100', 'Zinsaufwendungen', -180000, -170000, -175000),
        ('ZS', 'Finanzergebnis', -155000, -150000, -153000),
        ('ZS', 'EBT', 1625000, 1810000, 1771000),
        # Steuern
        ('7700', 'Ertragsteuern', -487500, -543000, -531300),
        ('ZS', 'Jahresüberschuss', 1137500, 1267000, 1239700),
    ]
    return pd.DataFrame(daten, columns=[COL_KONTO, COL_BEZEICHNUNG, COL_IST, COL_PLAN, COL_FORECAST])


def erzeuge_demo_ukv() -> pd.DataFrame:
    """Erzeugt realistische UKV-Demodaten."""
    daten = [
        # Umsatzerlöse
        ('4000', 'Umsatzerlöse Inland', 8500000, 8000000, 8300000),
        ('4100', 'Umsatzerlöse Export', 3200000, 3500000, 3400000),
        ('4200', 'Erlösschmälerungen', -350000, -300000, -320000),
        ('ZS', 'Umsatzerlöse (netto)', 11350000, 11200000, 11380000),
        # Herstellungskosten des Umsatzes
        ('5000', 'Materialeinzelkosten', -3200000, -3050000, -3150000),
        ('5100', 'Fertigungseinzelkosten', -1400000, -1350000, -1380000),
        ('5200', 'Fertigungsgemeinkosten', -850000, -800000, -830000),
        ('5300', 'Materialgemeinkosten', -480000, -450000, -470000),
        ('ZS', 'Herstellungskosten des Umsatzes', -5930000, -5650000, -5830000),
        ('ZS', 'Bruttoergebnis / DB1', 5420000, 5550000, 5550000),
        # Vertriebskosten
        ('6000', 'Vertriebspersonalkosten', -980000, -950000, -960000),
        ('6100', 'Werbung und Marketing', -210000, -200000, -205000),
        ('6200', 'Vertriebsgemeinkosten', -320000, -300000, -310000),
        ('ZS', 'Vertriebskosten', -1510000, -1450000, -1475000),
        # Verwaltungskosten
        ('6300', 'Verwaltungspersonalkosten', -650000, -630000, -640000),
        ('6400', 'IT und Bürokosten', -180000, -160000, -170000),
        ('6500', 'Sonstige Verwaltungskosten', -130000, -120000, -125000),
        ('ZS', 'Verwaltungskosten', -960000, -910000, -935000),
        ('ZS', 'DB2', 2950000, 3190000, 3140000),
        # Sonstige Erträge/Aufwendungen
        ('6700', 'Sonstige betriebliche Erträge', 180000, 150000, 165000),
        ('6800', 'Sonstige betriebliche Aufwendungen', -265000, -250000, -255000),
        ('ZS', 'EBITDA', 2865000, 3090000, 3050000),
        # Abschreibungen
        ('6900', 'Abschreibungen', -500000, -470000, -485000),
        ('ZS', 'EBIT', 2365000, 2620000, 2565000),
        # Finanzergebnis
        ('7000', 'Zinserträge', 25000, 20000, 22000),
        ('7100', 'Zinsaufwendungen', -180000, -170000, -175000),
        ('ZS', 'Finanzergebnis', -155000, -150000, -153000),
        ('ZS', 'EBT', 2210000, 2470000, 2412000),
        # Steuern
        ('7700', 'Ertragsteuern', -663000, -741000, -723600),
        ('ZS', 'Jahresüberschuss', 1547000, 1729000, 1688400),
    ]
    return pd.DataFrame(daten, columns=[COL_KONTO, COL_BEZEICHNUNG, COL_IST, COL_PLAN, COL_FORECAST])


if df_gkv is None:
    df_gkv = erzeuge_demo_gkv()
    df_ukv = erzeuge_demo_ukv()
    print('Demo-Daten erzeugt (GKV & UKV).\n')

print('=== GKV (Gesamtkostenverfahren) ===')
display(df_gkv)
print(f'\n=== UKV (Umsatzkostenverfahren) ===')
display(df_ukv)

## 4. Zwischensummen & Kennzahlen extrahieren

Die Zwischensummen (Konto = `ZS`) werden als zentrale Steuerungsgrößen extrahiert.

In [ ]:
def extrahiere_zwischensummen(df: pd.DataFrame) -> pd.DataFrame:
    """Extrahiert alle Zwischensummen-Zeilen aus der GuV."""
    mask = df[COL_KONTO].astype(str).str.upper() == 'ZS'
    zs = df.loc[mask, [COL_BEZEICHNUNG, COL_IST, COL_PLAN, COL_FORECAST]].copy()
    zs = zs.set_index(COL_BEZEICHNUNG)
    return zs


def hole_kennzahl(zs: pd.DataFrame, suchbegriffe: list, spalte: str = COL_IST) -> float:
    """Sucht eine Kennzahl anhand von Teilstrings in der Bezeichnung."""
    for begriff in suchbegriffe:
        treffer = zs.index[zs.index.str.contains(begriff, case=False, na=False)]
        if len(treffer) > 0:
            return float(zs.loc[treffer[0], spalte])
    return np.nan


# Zwischensummen extrahieren
zs_gkv = extrahiere_zwischensummen(df_gkv)
zs_ukv = extrahiere_zwischensummen(df_ukv)

print('=== Zwischensummen GKV ===')
display(zs_gkv.style.format('{:,.0f}'))
print('\n=== Zwischensummen UKV ===')
display(zs_ukv.style.format('{:,.0f}'))

## 5. Kennzahlenberechnung

Berechnung aller relevanten Steuerungskennzahlen für **Ist**, **Plan** und **Forecast**.

In [ ]:
# Suchbegriffe für die Kennzahlen-Zuordnung
KENNZAHL_MAPPING = {
    'Umsatzerlöse': ['Umsatzerlöse (netto)', 'Umsatzerlöse'],
    'Gesamtleistung': ['Gesamtleistung'],
    'Materialaufwand': ['Materialaufwand', 'Herstellungskosten'],
    'DB1': ['DB1', 'Rohertrag', 'Bruttoergebnis'],
    'Personalaufwand': ['Personalaufwand', 'Vertriebskosten'],
    'DB2': ['DB2'],
    'EBITDA': ['EBITDA'],
    'EBIT': ['EBIT'],
    'EBT': ['EBT'],
    'Jahresüberschuss': ['Jahresüberschuss'],
}


def berechne_kennzahlen(zs: pd.DataFrame, label: str) -> pd.DataFrame:
    """Berechnet alle Kennzahlen aus den Zwischensummen."""
    ergebnis = {}

    for name, suchbegriffe in KENNZAHL_MAPPING.items():
        ergebnis[name] = {
            COL_IST: hole_kennzahl(zs, suchbegriffe, COL_IST),
            COL_PLAN: hole_kennzahl(zs, suchbegriffe, COL_PLAN),
            COL_FORECAST: hole_kennzahl(zs, suchbegriffe, COL_FORECAST),
        }

    df_kz = pd.DataFrame(ergebnis).T

    # Umsatz als Referenz für Quoten
    umsatz_ist = df_kz.loc['Umsatzerlöse', COL_IST]
    umsatz_plan = df_kz.loc['Umsatzerlöse', COL_PLAN]
    umsatz_fc = df_kz.loc['Umsatzerlöse', COL_FORECAST]

    # Abweichungen Plan-Ist
    df_kz['Abw. Ist-Plan (abs)'] = df_kz[COL_IST] - df_kz[COL_PLAN]
    df_kz['Abw. Ist-Plan (%)'] = np.where(
        df_kz[COL_PLAN] != 0,
        (df_kz[COL_IST] - df_kz[COL_PLAN]) / df_kz[COL_PLAN].abs() * 100,
        0
    )

    # Abweichungen Forecast-Plan
    df_kz['Abw. FC-Plan (abs)'] = df_kz[COL_FORECAST] - df_kz[COL_PLAN]
    df_kz['Abw. FC-Plan (%)'] = np.where(
        df_kz[COL_PLAN] != 0,
        (df_kz[COL_FORECAST] - df_kz[COL_PLAN]) / df_kz[COL_PLAN].abs() * 100,
        0
    )

    df_kz.index.name = f'Kennzahl ({label})'
    return df_kz


kz_gkv = berechne_kennzahlen(zs_gkv, 'GKV')
kz_ukv = berechne_kennzahlen(zs_ukv, 'UKV')

print('=== Kennzahlen GKV ===')
display(kz_gkv.style.format('{:,.0f}').background_gradient(
    subset=['Abw. Ist-Plan (%)'], cmap='RdYlGn', vmin=-15, vmax=15
))
print('\n=== Kennzahlen UKV ===')
display(kz_ukv.style.format('{:,.0f}').background_gradient(
    subset=['Abw. Ist-Plan (%)'], cmap='RdYlGn', vmin=-15, vmax=15
))

## 6. Quoten & Rentabilitätskennzahlen

In [ ]:
def berechne_quoten(kz: pd.DataFrame, label: str) -> pd.DataFrame:
    """Berechnet Quoten bezogen auf den Umsatz."""
    umsatz = kz.loc['Umsatzerlöse', [COL_IST, COL_PLAN, COL_FORECAST]]

    quoten_daten = {}

    kennzahlen_fuer_quoten = [
        ('Materialquote', 'Materialaufwand'),
        ('Personalquote', 'Personalaufwand'),
        ('DB1-Marge', 'DB1'),
        ('DB2-Marge', 'DB2'),
        ('EBITDA-Marge', 'EBITDA'),
        ('EBIT-Marge', 'EBIT'),
        ('EBT-Marge', 'EBT'),
        ('Umsatzrentabilität', 'Jahresüberschuss'),
    ]

    for quotenname, kz_name in kennzahlen_fuer_quoten:
        if kz_name in kz.index:
            werte = kz.loc[kz_name, [COL_IST, COL_PLAN, COL_FORECAST]]
            quoten_daten[quotenname] = {
                f'{COL_IST} (%)': werte[COL_IST] / umsatz[COL_IST] * 100 if umsatz[COL_IST] != 0 else 0,
                f'{COL_PLAN} (%)': werte[COL_PLAN] / umsatz[COL_PLAN] * 100 if umsatz[COL_PLAN] != 0 else 0,
                f'{COL_FORECAST} (%)': werte[COL_FORECAST] / umsatz[COL_FORECAST] * 100 if umsatz[COL_FORECAST] != 0 else 0,
            }

    df_quoten = pd.DataFrame(quoten_daten).T
    df_quoten.index.name = f'Quote ({label})'
    return df_quoten


quoten_gkv = berechne_quoten(kz_gkv, 'GKV')
quoten_ukv = berechne_quoten(kz_ukv, 'UKV')

print('=== Quoten & Margen GKV (in %) ===')
display(quoten_gkv.style.format('{:.1f}%').background_gradient(cmap='RdYlGn', vmin=-5, vmax=30))
print('\n=== Quoten & Margen UKV (in %) ===')
display(quoten_ukv.style.format('{:.1f}%').background_gradient(cmap='RdYlGn', vmin=-5, vmax=30))

## 7. Detaillierte Abweichungsanalyse

In [ ]:
def abweichungsanalyse(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Erstellt eine detaillierte Abweichungsanalyse für alle Positionen."""
    analyse = df[[COL_BEZEICHNUNG, COL_IST, COL_PLAN, COL_FORECAST]].copy()
    analyse = analyse.set_index(COL_BEZEICHNUNG)

    # Plan-Ist Abweichung
    analyse['Ist-Plan (abs)'] = analyse[COL_IST] - analyse[COL_PLAN]
    analyse['Ist-Plan (%)'] = np.where(
        analyse[COL_PLAN] != 0,
        (analyse[COL_IST] - analyse[COL_PLAN]) / analyse[COL_PLAN].abs() * 100,
        0
    )

    # Forecast-Plan Abweichung
    analyse['FC-Plan (abs)'] = analyse[COL_FORECAST] - analyse[COL_PLAN]
    analyse['FC-Plan (%)'] = np.where(
        analyse[COL_PLAN] != 0,
        (analyse[COL_FORECAST] - analyse[COL_PLAN]) / analyse[COL_PLAN].abs() * 100,
        0
    )

    # Ist-Forecast Abweichung
    analyse['Ist-FC (abs)'] = analyse[COL_IST] - analyse[COL_FORECAST]
    analyse['Ist-FC (%)'] = np.where(
        analyse[COL_FORECAST] != 0,
        (analyse[COL_IST] - analyse[COL_FORECAST]) / analyse[COL_FORECAST].abs() * 100,
        0
    )

    analyse.index.name = f'Position ({label})'
    return analyse


abw_gkv = abweichungsanalyse(df_gkv, 'GKV')
abw_ukv = abweichungsanalyse(df_ukv, 'UKV')

print('=== Abweichungsanalyse GKV ===')
display(abw_gkv.style.format({
    COL_IST: '{:,.0f}', COL_PLAN: '{:,.0f}', COL_FORECAST: '{:,.0f}',
    'Ist-Plan (abs)': '{:,.0f}', 'Ist-Plan (%)': '{:+.1f}%',
    'FC-Plan (abs)': '{:,.0f}', 'FC-Plan (%)': '{:+.1f}%',
    'Ist-FC (abs)': '{:,.0f}', 'Ist-FC (%)': '{:+.1f}%',
}).background_gradient(subset=['Ist-Plan (%)'], cmap='RdYlGn', vmin=-15, vmax=15))

print('\n=== Abweichungsanalyse UKV ===')
display(abw_ukv.style.format({
    COL_IST: '{:,.0f}', COL_PLAN: '{:,.0f}', COL_FORECAST: '{:,.0f}',
    'Ist-Plan (abs)': '{:,.0f}', 'Ist-Plan (%)': '{:+.1f}%',
    'FC-Plan (abs)': '{:,.0f}', 'FC-Plan (%)': '{:+.1f}%',
    'Ist-FC (abs)': '{:,.0f}', 'Ist-FC (%)': '{:+.1f}%',
}).background_gradient(subset=['Ist-Plan (%)'], cmap='RdYlGn', vmin=-15, vmax=15))

---

## 8. Visualisierungen

### 8.1 Wasserfall-Diagramm: Von Umsatz zum Jahresüberschuss

In [ ]:
def wasserfall_guv(kz: pd.DataFrame, titel: str):
    """Erstellt ein Wasserfall-Diagramm der GuV-Kennzahlen (Ist)."""
    stufen = ['Umsatzerlöse', 'DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']
    verfuegbar = [s for s in stufen if s in kz.index]

    werte_ist = [kz.loc[s, COL_IST] for s in verfuegbar]

    fig, ax = plt.subplots(figsize=(14, 7))

    # Differenzen berechnen für Wasserfall-Effekt
    deltas = [werte_ist[0]]  # Erster Balken = absoluter Wert
    for i in range(1, len(werte_ist)):
        deltas.append(werte_ist[i] - werte_ist[i - 1])

    bottoms = [0]
    for i in range(1, len(werte_ist)):
        bottoms.append(werte_ist[i - 1])

    farben = []
    for i, d in enumerate(deltas):
        if i == 0:
            farben.append(FARBEN['ist'])
        elif i == len(deltas) - 1:
            farben.append(FARBEN['highlight'])
        elif d >= 0:
            farben.append(FARBEN['positiv'])
        else:
            farben.append(FARBEN['negativ'])

    bars = ax.bar(verfuegbar, deltas, bottom=bottoms, color=farben,
                  edgecolor='white', linewidth=1.5, width=0.6)

    # Werte beschriften
    for bar, wert in zip(bars, werte_ist):
        ax.text(bar.get_x() + bar.get_width() / 2, wert + max(werte_ist) * 0.02,
                f'{wert / 1e6:.2f} Mio', ha='center', va='bottom',
                fontweight='bold', fontsize=10)

    # Verbindungslinien
    for i in range(len(werte_ist) - 1):
        ax.plot([i + 0.3, i + 0.7], [werte_ist[i], werte_ist[i]],
                color='gray', linewidth=0.8, linestyle='--')

    ax.set_title(f'Wasserfall: {titel}', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Betrag (€)', fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x / 1e6:.1f} Mio'))
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


wasserfall_guv(kz_gkv, 'GuV nach GKV (Ist-Werte)')
wasserfall_guv(kz_ukv, 'GuV nach UKV (Ist-Werte)')

### 8.2 Ist vs. Plan vs. Forecast – Balkendiagramm

In [ ]:
def balken_ist_plan_fc(kz: pd.DataFrame, titel: str):
    """Gruppiertes Balkendiagramm: Ist / Plan / Forecast für alle Kennzahlen."""
    stufen = ['Umsatzerlöse', 'DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']
    verfuegbar = [s for s in stufen if s in kz.index]

    ist_werte = [kz.loc[s, COL_IST] for s in verfuegbar]
    plan_werte = [kz.loc[s, COL_PLAN] for s in verfuegbar]
    fc_werte = [kz.loc[s, COL_FORECAST] for s in verfuegbar]

    x = np.arange(len(verfuegbar))
    breite = 0.25

    fig, ax = plt.subplots(figsize=(16, 7))
    b1 = ax.bar(x - breite, ist_werte, breite, label='Ist', color=FARBEN['ist'],
                edgecolor='white', linewidth=0.5)
    b2 = ax.bar(x, plan_werte, breite, label='Plan', color=FARBEN['plan'],
                edgecolor='white', linewidth=0.5)
    b3 = ax.bar(x + breite, fc_werte, breite, label='Forecast', color=FARBEN['forecast'],
                edgecolor='white', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(verfuegbar, rotation=0)
    ax.set_title(f'Ist vs. Plan vs. Forecast – {titel}', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Betrag (€)', fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x / 1e6:.1f} Mio'))
    ax.legend(fontsize=12, loc='upper right')
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()


balken_ist_plan_fc(kz_gkv, 'GKV')
balken_ist_plan_fc(kz_ukv, 'UKV')

### 8.3 Plan-Ist-Abweichungen (absolut & prozentual)

In [ ]:
def abweichungs_chart(kz: pd.DataFrame, titel: str):
    """Horizontales Balkendiagramm der Plan-Ist-Abweichungen."""
    stufen = ['Umsatzerlöse', 'DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']
    verfuegbar = [s for s in stufen if s in kz.index]

    abw_abs = [kz.loc[s, 'Abw. Ist-Plan (abs)'] for s in verfuegbar]
    abw_pct = [kz.loc[s, 'Abw. Ist-Plan (%)'] for s in verfuegbar]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # Absolute Abweichung
    farben1 = [FARBEN['positiv'] if v >= 0 else FARBEN['negativ'] for v in abw_abs]
    bars1 = ax1.barh(verfuegbar, abw_abs, color=farben1, edgecolor='white', height=0.5)
    ax1.set_title(f'Ist-Plan Abweichung (absolut) – {titel}', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Abweichung (€)')
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x / 1e3:.0f} T€'))
    ax1.axvline(x=0, color='black', linewidth=0.8)
    for bar, val in zip(bars1, abw_abs):
        ax1.text(val + (max(abw_abs) - min(abw_abs)) * 0.02 * (1 if val >= 0 else -1),
                 bar.get_y() + bar.get_height() / 2,
                 f'{val / 1e3:+,.0f} T€', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')

    # Prozentuale Abweichung
    farben2 = [FARBEN['positiv'] if v >= 0 else FARBEN['negativ'] for v in abw_pct]
    bars2 = ax2.barh(verfuegbar, abw_pct, color=farben2, edgecolor='white', height=0.5)
    ax2.set_title(f'Ist-Plan Abweichung (prozentual) – {titel}', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Abweichung (%)')
    ax2.axvline(x=0, color='black', linewidth=0.8)
    for bar, val in zip(bars2, abw_pct):
        ax2.text(val + (max(abw_pct) - min(abw_pct)) * 0.03 * (1 if val >= 0 else -1),
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:+.1f}%', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()


abweichungs_chart(kz_gkv, 'GKV')
abweichungs_chart(kz_ukv, 'UKV')

### 8.4 Margen-Übersicht (DB1, DB2, EBITDA, EBIT, Umsatzrentabilität)

In [ ]:
def margen_chart(quoten: pd.DataFrame, titel: str):
    """Gruppiertes Balkendiagramm der Margen."""
    margen_namen = ['DB1-Marge', 'DB2-Marge', 'EBITDA-Marge', 'EBIT-Marge', 'Umsatzrentabilität']
    verfuegbar = [m for m in margen_namen if m in quoten.index]

    ist_vals = [quoten.loc[m, f'{COL_IST} (%)'] for m in verfuegbar]
    plan_vals = [quoten.loc[m, f'{COL_PLAN} (%)'] for m in verfuegbar]
    fc_vals = [quoten.loc[m, f'{COL_FORECAST} (%)'] for m in verfuegbar]

    x = np.arange(len(verfuegbar))
    breite = 0.25

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.bar(x - breite, ist_vals, breite, label='Ist', color=FARBEN['ist'])
    ax.bar(x, plan_vals, breite, label='Plan', color=FARBEN['plan'])
    ax.bar(x + breite, fc_vals, breite, label='Forecast', color=FARBEN['forecast'])

    # Werte beschriften
    for i, (ist, plan, fc) in enumerate(zip(ist_vals, plan_vals, fc_vals)):
        ax.text(i - breite, ist + 0.5, f'{ist:.1f}%', ha='center', fontsize=9, fontweight='bold')
        ax.text(i, plan + 0.5, f'{plan:.1f}%', ha='center', fontsize=9, fontweight='bold')
        ax.text(i + breite, fc + 0.5, f'{fc:.1f}%', ha='center', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(verfuegbar, rotation=0)
    ax.set_title(f'Margen-Übersicht – {titel}', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Marge (%)', fontsize=12)
    ax.legend(fontsize=12, loc='upper right')
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()


margen_chart(quoten_gkv, 'GKV')
margen_chart(quoten_ukv, 'UKV')

### 8.5 Kostenquoten-Vergleich (Material- & Personalquote)

In [ ]:
def kostenquoten_chart(quoten: pd.DataFrame, titel: str):
    """Kostenquoten als gestapeltes Balkendiagramm."""
    quoten_namen = ['Materialquote', 'Personalquote']
    verfuegbar = [q for q in quoten_namen if q in quoten.index]

    if not verfuegbar:
        print(f'Keine Kostenquoten für {titel} verfügbar.')
        return

    szenarien = [COL_IST, COL_PLAN, COL_FORECAST]
    farb_map = {'Materialquote': '#E57373', 'Personalquote': '#64B5F6'}

    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(szenarien))
    breite = 0.4
    bottom = np.zeros(len(szenarien))

    for q_name in verfuegbar:
        werte = [abs(quoten.loc[q_name, f'{sz} (%)']) for sz in szenarien]
        bars = ax.bar(x, werte, breite, bottom=bottom, label=q_name,
                       color=farb_map.get(q_name, FARBEN['neutral']), edgecolor='white')
        for bar, val, bot in zip(bars, werte, bottom):
            if val > 2:
                ax.text(bar.get_x() + bar.get_width() / 2, bot + val / 2,
                        f'{val:.1f}%', ha='center', va='center',
                        fontweight='bold', fontsize=11, color='white')
        bottom += np.array(werte)

    ax.set_xticks(x)
    ax.set_xticklabels(szenarien)
    ax.set_title(f'Kostenquoten am Umsatz – {titel}', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Quote (%)', fontsize=12)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


kostenquoten_chart(quoten_gkv, 'GKV')
kostenquoten_chart(quoten_ukv, 'UKV')

### 8.6 Forecast vs. Plan Abweichungen

In [ ]:
def fc_plan_abweichung_chart(kz: pd.DataFrame, titel: str):
    """Vergleich Forecast vs. Plan als Abweichungsdiagramm."""
    stufen = ['Umsatzerlöse', 'DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']
    verfuegbar = [s for s in stufen if s in kz.index]

    abw_pct = [kz.loc[s, 'Abw. FC-Plan (%)'] for s in verfuegbar]

    fig, ax = plt.subplots(figsize=(14, 6))

    farben = [FARBEN['positiv'] if v >= 0 else FARBEN['negativ'] for v in abw_pct]
    bars = ax.bar(verfuegbar, abw_pct, color=farben, edgecolor='white',
                  width=0.5, linewidth=1.5)

    for bar, val in zip(bars, abw_pct):
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + (0.3 if val >= 0 else -0.3),
                f'{val:+.1f}%', ha='center',
                va='bottom' if val >= 0 else 'top',
                fontweight='bold', fontsize=10)

    ax.set_title(f'Forecast vs. Plan Abweichung (%) – {titel}', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Abweichung (%)', fontsize=12)
    ax.axhline(y=0, color='black', linewidth=0.8)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


fc_plan_abweichung_chart(kz_gkv, 'GKV')
fc_plan_abweichung_chart(kz_ukv, 'UKV')

### 8.7 Dashboard: Kennzahlen-Übersicht auf einen Blick

In [ ]:
def kennzahlen_dashboard(kz: pd.DataFrame, quoten: pd.DataFrame, titel: str):
    """Kompaktes KPI-Dashboard mit den wichtigsten Steuerungsgrößen."""
    kpis = [
        ('Umsatzerlöse', 'Umsatzerlöse', 'abs'),
        ('DB1', 'DB1', 'abs'),
        ('DB1-Marge', 'DB1-Marge', 'pct'),
        ('DB2', 'DB2', 'abs'),
        ('EBITDA', 'EBITDA', 'abs'),
        ('EBITDA-Marge', 'EBITDA-Marge', 'pct'),
        ('EBIT', 'EBIT', 'abs'),
        ('Jahresüberschuss', 'Jahresüberschuss', 'abs'),
        ('Umsatzrentabilität', 'Umsatzrentabilität', 'pct'),
    ]

    fig, axes = plt.subplots(3, 3, figsize=(18, 12))
    fig.suptitle(f'KPI-Dashboard – {titel}', fontsize=20, fontweight='bold', y=1.02)

    for idx, (label, key, typ) in enumerate(kpis):
        ax = axes[idx // 3][idx % 3]
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.axis('off')

        if typ == 'abs' and key in kz.index:
            ist_val = kz.loc[key, COL_IST]
            plan_val = kz.loc[key, COL_PLAN]
            abw = ist_val - plan_val
            abw_pct = (abw / abs(plan_val) * 100) if plan_val != 0 else 0

            ax.text(5, 8, label, ha='center', va='center', fontsize=13, fontweight='bold',
                    color='#333333')
            ax.text(5, 5.5, f'{ist_val / 1e6:,.2f} Mio €', ha='center', va='center',
                    fontsize=20, fontweight='bold', color=FARBEN['ist'])
            abw_farbe = FARBEN['positiv'] if abw >= 0 else FARBEN['negativ']
            ax.text(5, 3, f'vs. Plan: {abw / 1e3:+,.0f} T€ ({abw_pct:+.1f}%)',
                    ha='center', va='center', fontsize=11, color=abw_farbe, fontweight='bold')
            ax.text(5, 1.5, f'Plan: {plan_val / 1e6:,.2f} Mio', ha='center', va='center',
                    fontsize=9, color=FARBEN['neutral'])

        elif typ == 'pct' and key in quoten.index:
            ist_val = quoten.loc[key, f'{COL_IST} (%)']
            plan_val = quoten.loc[key, f'{COL_PLAN} (%)']
            abw = ist_val - plan_val

            ax.text(5, 8, label, ha='center', va='center', fontsize=13, fontweight='bold',
                    color='#333333')
            ax.text(5, 5.5, f'{ist_val:.1f}%', ha='center', va='center',
                    fontsize=24, fontweight='bold', color=FARBEN['ist'])
            abw_farbe = FARBEN['positiv'] if abw >= 0 else FARBEN['negativ']
            ax.text(5, 3, f'vs. Plan: {abw:+.1f} PP',
                    ha='center', va='center', fontsize=11, color=abw_farbe, fontweight='bold')
            ax.text(5, 1.5, f'Plan: {plan_val:.1f}%', ha='center', va='center',
                    fontsize=9, color=FARBEN['neutral'])

        # Rahmen
        ax.add_patch(plt.Rectangle((0.3, 0.5), 9.4, 9, fill=False,
                                    edgecolor='#E0E0E0', linewidth=2, zorder=0))

    plt.tight_layout()
    plt.show()


kennzahlen_dashboard(kz_gkv, quoten_gkv, 'GKV')
kennzahlen_dashboard(kz_ukv, quoten_ukv, 'UKV')

---

## 9. GKV vs. UKV Vergleich

In [ ]:
def vergleich_gkv_ukv(kz_gkv: pd.DataFrame, kz_ukv: pd.DataFrame):
    """Vergleicht zentrale Kennzahlen zwischen GKV und UKV."""
    gemeinsame = ['Umsatzerlöse', 'DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']
    verfuegbar = [s for s in gemeinsame if s in kz_gkv.index and s in kz_ukv.index]

    gkv_ist = [kz_gkv.loc[s, COL_IST] for s in verfuegbar]
    ukv_ist = [kz_ukv.loc[s, COL_IST] for s in verfuegbar]

    x = np.arange(len(verfuegbar))
    breite = 0.35

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.bar(x - breite / 2, gkv_ist, breite, label='GKV (Ist)', color=FARBEN['ist'],
           edgecolor='white')
    ax.bar(x + breite / 2, ukv_ist, breite, label='UKV (Ist)', color=FARBEN['forecast'],
           edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(verfuegbar, rotation=0)
    ax.set_title('Vergleich GKV vs. UKV (Ist-Werte)', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Betrag (€)', fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x / 1e6:.1f} Mio'))
    ax.legend(fontsize=12)
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()


vergleich_gkv_ukv(kz_gkv, kz_ukv)

---

## 10. Zusammenfassung & Export

Abschließende Zusammenfassung und optionaler Export der Ergebnisse.

In [ ]:
def zusammenfassung(kz: pd.DataFrame, quoten: pd.DataFrame, label: str):
    """Gibt eine kompakte textuelle Zusammenfassung aus."""
    print(f'\n{"=" * 60}')
    print(f' ZUSAMMENFASSUNG – {label}')
    print(f'{"=" * 60}\n')

    umsatz = kz.loc['Umsatzerlöse', COL_IST] if 'Umsatzerlöse' in kz.index else 0
    print(f'  Umsatzerlöse (Ist):       {umsatz / 1e6:>10,.2f} Mio €')

    for name in ['DB1', 'DB2', 'EBITDA', 'EBIT', 'EBT', 'Jahresüberschuss']:
        if name in kz.index:
            ist = kz.loc[name, COL_IST]
            abw = kz.loc[name, 'Abw. Ist-Plan (%)']
            print(f'  {name:<24s} {ist / 1e6:>10,.2f} Mio €  (Plan-Abw.: {abw:+.1f}%)')

    print(f'\n  --- Margen ---')
    for name in ['DB1-Marge', 'DB2-Marge', 'EBITDA-Marge', 'EBIT-Marge', 'Umsatzrentabilität']:
        if name in quoten.index:
            val = quoten.loc[name, f'{COL_IST} (%)']
            print(f'  {name:<24s} {val:>10.1f}%')

    print(f'\n  --- Kostenquoten ---')
    for name in ['Materialquote', 'Personalquote']:
        if name in quoten.index:
            val = abs(quoten.loc[name, f'{COL_IST} (%)'])
            print(f'  {name:<24s} {val:>10.1f}%')

    print()


zusammenfassung(kz_gkv, quoten_gkv, 'GKV')
zusammenfassung(kz_ukv, quoten_ukv, 'UKV')

In [ ]:
# Optionaler Export der Ergebnisse nach Excel
EXPORT_PFAD = 'GuV_Analyse_Ergebnis.xlsx'
EXPORT_AKTIVIERT = False  # Auf True setzen für Export

if EXPORT_AKTIVIERT:
    with pd.ExcelWriter(EXPORT_PFAD, engine='openpyxl') as writer:
        kz_gkv.to_excel(writer, sheet_name='Kennzahlen_GKV')
        kz_ukv.to_excel(writer, sheet_name='Kennzahlen_UKV')
        quoten_gkv.to_excel(writer, sheet_name='Quoten_GKV')
        quoten_ukv.to_excel(writer, sheet_name='Quoten_UKV')
        abw_gkv.to_excel(writer, sheet_name='Abweichung_GKV')
        abw_ukv.to_excel(writer, sheet_name='Abweichung_UKV')
    print(f'Ergebnisse exportiert nach: {EXPORT_PFAD}')
else:
    print('Export deaktiviert. Setze EXPORT_AKTIVIERT = True zum Exportieren.')

---

## Hinweise zur Nutzung

1. **Excel-Datei anpassen**: Passe in Abschnitt 2 den `EXCEL_PFAD` und die Blattnamen (`BLATT_GKV`, `BLATT_UKV`) an
2. **Spalten umbenennen**: Falls deine Spalten anders heißen, ändere die `COL_*` Variablen in Abschnitt 2
3. **Zwischensummen-Erkennung**: Zeilen mit `Konto = 'ZS'` werden als Zwischensummen erkannt. Passe dies ggf. an dein Format an
4. **Export**: Setze `EXPORT_AKTIVIERT = True` in der letzten Zelle für einen Excel-Export der Ergebnisse
5. **Abhängigkeiten**: `pandas`, `openpyxl`, `matplotlib`, `seaborn`, `numpy`